# Experimento 1 — SVM sem balanceamento utilizando 4 variáveis

Este experimento utiliza o algoritmo SVM (Support Vector Machine) aplicado ao dataset de qualidade hídrica, mantendo as mesmas condições experimentais utilizadas nos experimentos anteriores para permitir comparação direta entre os algoritmos.

As variáveis utilizadas como entrada foram:

- `Temperature (cel)`
- `Orthophosphate (mg/l)`
- `Country`
- `Waterbody Type`

A variável alvo (`y`) utilizada para previsão foi:

- `conama_status`

O dataset foi dividido em:

- 80% para treino
- 20% para teste

A separação foi realizada com `train_test_split`, utilizando `random_state=42` e `stratify=y`.

Nenhuma técnica de balanceamento foi aplicada. O objetivo é avaliar o comportamento do SVM no mesmo cenário dos modelos anteriores, alterando apenas o algoritmo.

In [ ]:
# IMPORTAÇÃO DE BIBLIOTECAS
import pandas as pd
import numpy as np
import os
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

SEED = 42

In [ ]:
# DETECÇÃO DE AMBIENTE
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Ambiente Google Colab detectado.")
    drive.mount('/content/drive')

    DATA_PATH = Path(
        "/content/drive/MyDrive/EDA_AquaSense/Dataset/processed/amostra_rotulada.parquet"
    )
else:
    print("Ambiente local/VS Code detectado.")

    DATA_PATH = Path("amostra_rotulada.parquet")

df = pd.read_parquet(DATA_PATH)

print("Dataset Parquet carregado com sucesso.")
print(f"Shape do dataset: {df.shape}")

df.head()

In [ ]:
# PREPARAÇÃO PARA MACHINE LEARNING
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

In [ ]:
# DEFINIÇÃO DE X E Y
X = df[
    [
        "Temperature (cel)",
        "Orthophosphate (mg/l)",
        "Country",
        "Waterbody Type"
    ]
]

y = df["conama_status"]

In [ ]:
# DIVISÃO TREINO/TESTE
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

print("Treino:", X_train.shape)
print("Teste:", X_test.shape)

In [ ]:
# PRÉ-PROCESSAMENTO
categorical_features = [
    "Country",
    "Waterbody Type"
]

numeric_features = [
    "Temperature (cel)",
    "Orthophosphate (mg/l)"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),
        (
            "num",
            StandardScaler(),
            numeric_features
        )
    ]
)

In [ ]:
# MODELO SVM
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            SVC(
                kernel="rbf",
                random_state=SEED
            )
        )
    ]
)

model.fit(X_train, y_train)

## Avaliação das Métricas de Treino

Antes da avaliação no conjunto de teste, também foram calculadas as métricas no conjunto de treino. Essa etapa permite observar se há diferença significativa entre treino e teste, o que pode indicar overfitting.

In [ ]:
# MÉTRICAS DE TREINO
y_train_pred = model.predict(X_train)

train_accuracy = accuracy_score(y_train, y_train_pred)

train_precision = precision_score(
    y_train,
    y_train_pred,
    average="weighted"
)

train_recall = recall_score(
    y_train,
    y_train_pred,
    average="weighted"
)

train_f1 = f1_score(
    y_train,
    y_train_pred,
    average="weighted"
)

train_confusion_matrix = confusion_matrix(
    y_train,
    y_train_pred
)

print("Train Accuracy:")
print(train_accuracy)

print("Train Precision:")
print(train_precision)

print("Train Recall:")
print(train_recall)

print("Train F1:")
print(train_f1)

print("Train Confusion Matrix:")
print(train_confusion_matrix)

In [ ]:
# MÉTRICAS DE TESTE
y_pred = model.predict(X_test)

test_accuracy = accuracy_score(y_test, y_pred)
test_precision = precision_score(y_test, y_pred, average="weighted")
test_recall = recall_score(y_test, y_pred, average="weighted")
test_f1 = f1_score(y_test, y_pred, average="weighted")

print("Accuracy:")
print(test_accuracy)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

## Resultados Obtidos — Experimento 1

A acurácia obtida no conjunto de teste foi de aproximadamente:

In [ ]:
0.742

Enquanto isso, a acurácia no conjunto de treino ficou em torno de:

In [ ]:
0.74


### Comparação com Experimentos Anteriores

| Métrica | Exp 1 — RF s/ bal | Exp 1 — LGBM s/ bal | Exp 1 — SVM s/ bal |
|--------------------|-------------------|----------------------|----------------------|
| Accuracy treino | 0.88 | 0.75 | 0.74 |
| Accuracy teste | 0.71 | 0.745 | 0.742 |
| Precision Atenção | 0.32 | 0.42 | 0.41 |
| Precision Boa | 0.33 | 0.39 | 0.38 |
| Precision Crítica | 0.10 | 0.17 | 0.00 |
| Precision Excelente | 0.80 | 0.79 | 0.77 |
| Recall Atenção | 0.20 | 0.21 | 0.22 |
| Recall Boa | 0.24 | 0.16 | 0.08 |
| Recall Crítica | 0.05 | 0.01 | 0.00 |
| Recall Excelente | 0.89 | 0.96 | 0.97 |
| F1 Atenção | 0.24 | 0.28 | 0.28 |
| F1 Boa | 0.28 | 0.23 | 0.13 |
| F1 Crítica | 0.06 | 0.02 | 0.00 |
| F1 Excelente | 0.84 | 0.87 | 0.86 |
| Overfitting | 0.17 | 0.005 | ~0.00 |

### Análise Comparativa

Os três algoritmos apresentaram comportamentos bastante distintos ao serem aplicados no mesmo cenário experimental.

O Random Forest apresentou desempenho mais equilibrado entre as classes, especialmente nas categorias minoritárias. Apesar da menor acurácia global, o modelo conseguiu identificar parte das amostras da classe `Crítica`, obtendo recall de 0.05. Isso indica maior sensibilidade para situações ambientais críticas, mesmo com desempenho geral inferior aos demais modelos.

O LightGBM apresentou a maior capacidade de generalização entre os experimentos realizados até o momento. A diferença mínima entre treino e teste demonstrou baixo overfitting, além de melhora na precision das classes minoritárias. Entretanto, o recall da classe `Crítica` permaneceu extremamente baixo, indicando que o modelo passou a prever essa classe em poucas ocasiões.

Já o SVM apresentou acurácia semelhante ao LightGBM e superior ao Random Forest, além de praticamente ausência de overfitting. O modelo concentrou fortemente seu aprendizado na classe majoritária `Excelente`, alcançando recall de 0.97 nessa categoria. Porém, o desempenho nas classes minoritárias foi significativamente comprometido, especialmente em `Crítica`, onde o recall foi igual a 0.00.

Os resultados evidenciam o impacto do desbalanceamento do dataset. Embora métricas globais como accuracy tenham permanecido elevadas, os modelos apresentaram dificuldades importantes na identificação das classes menos representadas.

### Conclusão

O Experimento 1 demonstrou que o SVM sem balanceamento possui boa capacidade de generalização e elevada performance na classificação da classe majoritária `Excelente`. Entretanto, o modelo falhou completamente na identificação da classe `Crítica`, tornando-se inadequado para aplicações ambientais que dependem da detecção de situações potencialmente perigosas.

Comparando os três algoritmos avaliados até o momento:

- O Random Forest apresentou maior equilíbrio entre as classes;
- O LightGBM apresentou melhor capacidade de generalização;
- O SVM apresentou maior concentração de desempenho na classe majoritária.

Os resultados obtidos reforçam que métricas globais não são suficientes para avaliação de modelos em problemas ambientais críticos. O próximo passo dos experimentos deverá investigar técnicas de balanceamento, como `class_weight="balanced"` e oversampling, buscando melhorar o recall das classes minoritárias sem comprometer a generalização dos modelos.